## Phase 2 ##
- Missile Data lookup
- Missile patterning 
- Geo Spaces and open sourced information 

In [7]:
# phase_2Jun3v3

# Scrape through missle databases to localize the data and find the right ones to use and make sure it works for use case 
# Need to also setup spacialized spaces

In [8]:
import numpy as np
import plotly.graph_objects as go

# --- 1. Physics Engine ---
def missile_dynamics(state, t, accel_cmd=np.zeros(3)):
    vel = state[3:6]
    gravity_accel = np.array([0.0, 0.0, -9.81])
    
    # Zeroing drag for the initial ballistic vacuum test at scale
    drag_accel = np.array([0.0, 0.0, 0.0]) 
    
    accel = gravity_accel + drag_accel + accel_cmd
    return np.concatenate((vel, accel))

def rk4_step(state, t, dt, derivatives_fn, accel_cmd=np.zeros(3)):
    k1 = derivatives_fn(state, t, accel_cmd)
    k2 = derivatives_fn(state + 0.5 * dt * k1, t + 0.5 * dt, accel_cmd)
    k3 = derivatives_fn(state + 0.5 * dt * k2, t + 0.5 * dt, accel_cmd)
    k4 = derivatives_fn(state + dt * k3, t + dt, accel_cmd)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

In [9]:
# --- 2. Geo Space Definitions (in meters) ---
# Space 1: 1,600km x 1,600km (Origin)
s1_min, s1_max = -800000, 800000 

# Space 2: 500km x 500km (Located 4,000km downrange on the X-axis)
s2_center_x = 4000000 
s2_half_side = 250000
s2_x_min, s2_x_max = s2_center_x - s2_half_side, s2_center_x + s2_half_side
s2_y_min, s2_y_max = -s2_half_side, s2_half_side

# --- 3. Initial Launch State ---
# Launching from the center of Space 1, aiming at the center of Space 2
# State: [x, y, z, vx, vy, vz]
# vx = 3800 m/s, vz = 5200 m/s yields a flight time of ~1060s and range of ~4000km
missile_state = np.array([0.0, 0.0, 0.0, 3800.0, 0.0, 5200.0]) 

dt = 1.0  # Increased time step to 1 second because the flight is long
total_time = 1100  
steps = int(total_time / dt)

missile_path = np.zeros((steps, 3))

# --- 4. Simulation Loop ---
for i in range(steps):
    missile_path[i] = missile_state[0:3]
    t = i * dt
    
    missile_state = rk4_step(missile_state, t, dt, missile_dynamics)
    
    # Stop recording if it hits the ground
    if missile_state[2] < 0 and i > 10: 
        missile_path = missile_path[:i] # Truncate array to actual flight time
        steps = i
        break

print(f"--- Surface-to-Surface Launch ---")
print(f"Flight Time: {steps * dt} seconds")
print(f"Impact Coordinates: X: {missile_path[-1,0]:.0f}m, Y: {missile_path[-1,1]:.0f}m")

--- Surface-to-Surface Launch ---
Flight Time: 1060.0 seconds
Impact Coordinates: X: 4024200m, Y: 0m


In [10]:
# --- 5. 3D Plotting at Scale ---
fig = go.Figure()

# Draw Space 1 (Green)
fig.add_trace(go.Mesh3d(
    x=[s1_min, s1_max, s1_max, s1_min],
    y=[s1_min, s1_min, s1_max, s1_max],
    z=[0, 0, 0, 0],
    color='rgba(0, 255, 0, 0.2)',
    name='Space 1 (Launch)'
))

# Draw Space 2 (Red)
fig.add_trace(go.Mesh3d(
    x=[s2_x_min, s2_x_max, s2_x_max, s2_x_min],
    y=[s2_y_min, s2_y_min, s2_y_max, s2_y_max],
    z=[0, 0, 0, 0],
    color='rgba(255, 0, 0, 0.2)',
    name='Space 2 (Target)'
))

# Plot Ballistic Trajectory
fig.add_trace(go.Scatter3d(
    x=missile_path[:,0], y=missile_path[:,1], z=missile_path[:,2],
    mode='lines', line=dict(color='orange', width=4), name='Ballistic Arc'
))

# Animated Missile Marker
fig.add_trace(go.Scatter3d(
    x=[missile_path[0,0]], y=[missile_path[0,1]], z=[missile_path[0,2]], 
    mode='markers', marker=dict(size=6, color='red'), name='Missile'
))

# Animation Frames
frame_skip = 20 # Skipping frames to keep animation fast over 1000+ seconds
frames = []
for k in range(0, steps, frame_skip):
    frames.append(go.Frame(
        data=[go.Scatter3d(x=[missile_path[k,0]], y=[missile_path[k,1]], z=[missile_path[k,2]])],
        traces=[3], # Update only the missile marker
        name=f'frame_{k}'
    ))
fig.frames = frames

fig.update_layout(
    title='Phase 2: Surface-to-Surface Macro Scale Launch',
    scene=dict(
        xaxis_title='X (meters)',
        yaxis_title='Y (meters)',
        zaxis_title='Altitude (meters)',
        aspectmode='data' # Forces axes to scale proportionally to real distance
    ),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=20, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )]
)

fig.show()